In [1]:
!pip install -U langgraph langchain langchain-openai

  Using cached langgraph-0.6.6-py3-none-any.whl.metadata (6.8 kB)
  Using cached langgraph_checkpoint-2.1.1-py3-none-any.whl.metadata (4.2 kB)
  Using cached langgraph_prebuilt-0.6.4-py3-none-any.whl.metadata (4.5 kB)
Using cached langgraph-0.6.6-py3-none-any.whl (153 kB)
Using cached langgraph_checkpoint-2.1.1-py3-none-any.whl (43 kB)
Using cached langgraph_prebuilt-0.6.4-py3-none-any.whl (28 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [langgraph]


In [23]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END,MessagesState
from langgraph.checkpoint.memory import InMemorySaver

In [11]:
import os

os.environ["OPENAI_API_KEY"] = "sk-60a9041004c34c4caffab1616e388989"
os.environ["OPENAI_BASE_URL"] = "https://dashscope.aliyuncs.com/compatible-mode/v1"
os.environ["OPENAI_MODEL"] = "qwen-turbo"
os.environ["EMBEDDING_MODEL"] = "text-embedding-v4"

In [12]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),  # 若走原生 OpenAI，可传 None/不传
    temperature=0.2,                        # 稳定输出
    timeout=1200,                             # 超时保护（秒）
    max_retries=2                           # 简单重试
)

In [16]:
# 2) 节点函数：接收 state，返回新增消息列表
def llm_node(state: MessagesState):
    resp = llm.invoke(state["messages"])   # 传入对话消息列表
    return {"messages": [resp]} 

In [17]:
graph = StateGraph(state_schema=MessagesState)

In [19]:
graph.add_node("llm", llm_node)
graph.add_edge(START, "llm")
graph.add_edge("llm", END)

In [25]:
memory = InMemorySaver()
app = graph.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "demo-thread-1"}}
app = app.with_config(config)

In [26]:
init_state = {"messages": [HumanMessage(content="你好，我是jaguarliu，请你简单自我介绍一下。")]}
result = app.invoke(init_state)
print(result["messages"][-1].content)

你好，JaguarLiu！很高兴认识你。我是通义千问，也就是Qwen，是阿里巴巴集团旗下的通义实验室自主研发的超大规模语言模型。我可以帮助你回答问题、创作文字、编程、逻辑推理等多种任务。如果你有任何问题或需要帮助，随时告诉我，我会尽力为你提供支持。希望我们能有愉快的交流！


In [27]:
user_messages = {"messages": [HumanMessage(content="你还记得我是谁吗")]}
result = app.invoke(user_messages)
print(result["messages"][-1].content)

你好，JaguarLiu！当然记得你，很高兴再次见到你。我是通义千问，也就是Qwen，一个由阿里巴巴集团旗下的通义实验室自主研发的超大规模语言模型。我能够帮助你回答问题、创作文字、编程、逻辑推理等多种任务。如果你有任何问题或需要帮助，随时告诉我，我会尽力为你提供支持。希望我们能有愉快的交流！


In [35]:
for update in app.stream({"messages": [HumanMessage(content="介绍一下 LangGraph")]},
                         stream_mode="updates"):
    for _, state in update.items():
        if "messages" in state:
            print(state["messages"][-1].content, end="")

你好，JaguarLiu！你已经多次询问 **LangGraph** 的介绍，我理解你是想更深入地了解它。下面我会为你做一个**全面、清晰、结构化**的介绍，帮助你更好地理解 LangGraph 是什么、它有什么功能、适合什么场景，以及它的技术特点。

---

## 🧠 什么是 LangGraph？

**LangGraph** 是一个用于构建和运行复杂语言模型（LLM）应用的框架，它通过图结构（Graph-based）的方式组织任务流程，使得开发者可以更灵活地设计多步骤、条件分支、循环等复杂的逻辑。

它的核心目标是让语言模型的应用更加**模块化、可扩展、可维护**，并支持与外部工具、API、数据库等的集成。

---

## 📌 核心概念

### 1. **节点（Node）**
- 每个节点代表一个独立的任务或处理单元。
- 可以是：
  - 调用语言模型（如 Qwen、GPT）
  - 执行外部 API 调用
  - 数据清洗或转换
  - 条件判断（如 if/else）

### 2. **边（Edge）**
- 定义节点之间的执行顺序或数据流向。
- 支持**条件边**（根据某些条件选择不同的路径）。

### 3. **图（Graph）**
- 由多个节点和边组成的有向图，表示整个应用的流程。
- 支持复杂的控制流，比如并行执行、循环、分支等。

---

## ✅ 主要功能

### 1. **流程编排**
- 通过图形化方式设计复杂的处理流程。
- 支持条件分支、循环、并行任务等。

### 2. **模块化开发**
- 每个节点可以独立开发、测试和部署。
- 提高代码的可维护性和复用性。

### 3. **状态管理**
- 在图中传递和存储状态信息，供后续节点使用。
- 例如：用户输入、中间结果、上下文信息等。

### 4. **与语言模型结合**
- 可以轻松集成大语言模型（如 Qwen、GPT、LLaMA 等）。
- 实现更智能的对话、推理、生成等任务。

---

## 🧩 适用场景

### 1. **多轮对话系统**
- 如客服机器人、智能助手等，需要处理多轮交互。

### 2. **数据处理流水线**
- 信息提取、内容生成、数据分析等需要多个步骤处理的任务。

### 3. **自动化工作流**
- 将多个工具或服务组合成一